# 5회차 · AI Agent Web Service MVP 완성 및 평가

**4회차 FastAPI 프로젝트에 Web UI를 추가**합니다. GPU 추론은 오직 RunPod에서 실행합니다.  
**수업 운영:** Web UI 작성 → FastAPI 연계 → 실행 → 정상/오류 평가 → 개선 → GitHub/Portfolio.

## STEP 1. 5회차 프로젝트 위치 확인

In [1]:
from pathlib import Path
import sys
BASE_DIR=Path("/workspace") if Path("/workspace").exists() else Path.cwd()
PROJECT_DIR=BASE_DIR/"05_agent_web_mvp"
assert (PROJECT_DIR/"app"/"agent.py").exists(), "05_agent_web_mvp.zip을 먼저 /workspace에 풀어 프로젝트를 준비하세요."
print("프로젝트 확인:",PROJECT_DIR)

프로젝트 확인: /workspace/05_agent_web_mvp


## STEP 2. Web UI의 HTML 작성 — `app/static/index.html`

In [2]:
from pathlib import Path
file_path = PROJECT_DIR / 'app/static/index.html'
file_path.parent.mkdir(parents=True, exist_ok=True)
if file_path.exists():
    print("기존 파일 보존 (덮어쓰지 않음):", file_path)

else:
    file_path.write_text(r'''
    <!doctype html>
    <html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
    <title>YU AI Agent Web MVP</title><link rel="stylesheet" href="/static/style.css"></head>
    <body><main class="wrap">
    <header><p class="eyebrow">RunPod A100 · Qwen3-8B · FastAPI</p><h1>AI Agent Web MVP</h1><p>질문에 따라 Custom Tool을 선택하고 실행합니다.</p></header>
    <section class="panel"><label for="question">질문 입력</label>
    <textarea id="question" rows="4" maxlength="1000" placeholder="예: 33명이 30시간씩 GPU를 사용하면 총 몇 GPU-hour야?"></textarea>
    <div class="actions"><button id="send">Agent 실행</button><button class="secondary" id="health">GPU 상태 확인</button></div>
    <p id="state" role="status" aria-live="polite">대기 중</p></section>
    <section class="grid"><article class="panel"><h2>최종 응답</h2><pre id="answer">아직 응답이 없습니다.</pre></article>
    <article class="panel"><h2>Agent 실행 정보</h2><pre id="meta">Tool 사용 여부, 처리시간</pre></article></section>
    <footer>교육용 데모 · 공개 인터넷에 인증 없이 노출하지 마세요.</footer>
    </main><script src="/static/app.js"></script></body></html>

    ''', encoding="utf-8")
print("확인:", file_path)


기존 파일 보존 (덮어쓰지 않음): /workspace/05_agent_web_mvp/app/static/index.html
확인: /workspace/05_agent_web_mvp/app/static/index.html


## STEP 3. CSS 스타일 작성 — `app/static/style.css`

In [3]:
from pathlib import Path
file_path = PROJECT_DIR / 'app/static/style.css'
file_path.parent.mkdir(parents=True, exist_ok=True)
if file_path.exists():
    print("기존 파일 보존 (덮어쓰지 않음):", file_path)
    print("수정은 VS Code에서 해당 파일을 직접 열어 진행하세요.")
else:
    file_path.write_text(r'''
    *{box-sizing:border-box}body{font-family:system-ui,-apple-system,"Malgun Gothic",sans-serif;background:#f6faff;color:#173253;margin:0}
    .wrap{max-width:1100px;margin:35px auto;padding:0 22px}header{background:#fff;border:1px solid #dce8f5;border-radius:18px;padding:30px;margin-bottom:20px}
    h1{font-size:32px;margin:8px 0}.eyebrow{font-weight:700;color:#1675b6;letter-spacing:.06em}.panel{background:#fff;border:1px solid #dce8f5;border-radius:16px;padding:23px;margin:15px 0;box-shadow:0 4px 18px #123a5510}
    label{display:block;font-weight:700;margin-bottom:10px}textarea{width:100%;border:1px solid #b9cfe1;border-radius:10px;padding:12px;font:inherit;resize:vertical}.actions{display:flex;gap:12px;margin-top:12px;flex-wrap:wrap}
    button{background:#217cbb;border:0;color:white;border-radius:10px;padding:12px 18px;font-weight:700;cursor:pointer}button:disabled{opacity:.6}.secondary{background:#e3f1fa;color:#1a5782}
    .grid{display:grid;grid-template-columns:1fr 1fr;gap:15px}pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#f5f9fe;padding:14px;border-radius:8px;min-height:100px;font:14px/1.65 Consolas,"Malgun Gothic",monospace}
    #state{font-size:14px;color:#326a92}footer{font-size:13px;color:#68829a;margin:24px 0}@media(max-width:760px){.grid{grid-template-columns:1fr}}

    ''', encoding="utf-8")
print("확인:", file_path)


기존 파일 보존 (덮어쓰지 않음): /workspace/05_agent_web_mvp/app/static/style.css
수정은 VS Code에서 해당 파일을 직접 열어 진행하세요.
확인: /workspace/05_agent_web_mvp/app/static/style.css


## STEP 4. JavaScript 작성 — `app/static/app.js`
`fetch("/agent")`로 질문을 보내고 응답·오류·실행 정보를 표시합니다.

In [4]:
from pathlib import Path
file_path = PROJECT_DIR / 'app/static/app.js'
file_path.parent.mkdir(parents=True, exist_ok=True)
if file_path.exists():
    print("기존 파일 보존 (덮어쓰지 않음):", file_path)
    print("수정은 VS Code에서 해당 파일을 직접 열어 진행하세요.")
else:
    file_path.write_text(r'''
    const el=id=>document.getElementById(id);
    async function fetchJSON(url, options={}) {
      const res=await fetch(url,options);
      const data=await res.json();
      if (!res.ok) throw new Error(typeof data.detail === "string" ? data.detail : `HTTP ${res.status}: ${JSON.stringify(data.detail)}`);
      return data;
    }
    async function runAgent() {
      const query=el("question").value.trim();
      if(!query){el("state").textContent="질문을 입력하세요.";return;}
      el("send").disabled=true;el("state").textContent="Agent 실행 중... (최초 모델 로딩 시 지연될 수 있음)";
      try {
        const data=await fetchJSON("/agent",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({query})});
        el("answer").textContent=data.final_answer||"답변 없음";
        el("meta").textContent=JSON.stringify({status:data.status,used_tool:data.used_tool,tool_name:data.tool_name,
          arguments:data.arguments,tool_result:data.tool_result,elapsed_ms:data.elapsed_ms},null,2);
        el("state").textContent="완료";
      } catch(e) {el("state").textContent="실행 오류";el("answer").textContent=e.message;}
      finally{el("send").disabled=false;}
    }
    async function checkHealth(){try{const data=await fetchJSON("/health");el("meta").textContent=JSON.stringify(data,null,2);
     el("state").textContent=data.cuda_available?"GPU 사용 가능":"CUDA GPU 없음 (Agent 실행 불가)";
    }catch(e){el("state").textContent=e.message;}}
    el("send").addEventListener("click",runAgent);el("health").addEventListener("click",checkHealth);
    el("question").addEventListener("keydown",e=>{if(e.key==="Enter"&&e.ctrlKey){e.preventDefault();runAgent();}});

    ''', encoding="utf-8")
print("확인:", file_path)


기존 파일 보존 (덮어쓰지 않음): /workspace/05_agent_web_mvp/app/static/app.js
수정은 VS Code에서 해당 파일을 직접 열어 진행하세요.
확인: /workspace/05_agent_web_mvp/app/static/app.js


## STEP 5. FastAPI에서 Web UI 제공하도록 수정 — `app/main.py`

In [5]:
from pathlib import Path
file_path = PROJECT_DIR / 'app/main.py'
file_path.parent.mkdir(parents=True, exist_ok=True)
if file_path.exists():
    print("기존 파일 보존 (덮어쓰지 않음):", file_path)

else:
    file_path.write_text(r'''
    """5회차: 4회차 FastAPI Backend에 HTML/CSS/JS 화면만 확장."""
    from pathlib import Path
    from fastapi import FastAPI, HTTPException
    from fastapi.responses import FileResponse
    from fastapi.staticfiles import StaticFiles
    from pydantic import BaseModel, Field, field_validator
    from .agent import run_agent
    from .llm import MODEL_ID, gpu_status
    from .tools import TOOL_DESCRIPTIONS

    STATIC_DIR = Path(__file__).parent / "static"
    app = FastAPI(title="YU AI Agent Web MVP", version="1.0.0")
    app.mount("/static", StaticFiles(directory=STATIC_DIR), name="static")

    class AgentRequest(BaseModel):
        query: str = Field(min_length=1, max_length=1000)

        @field_validator("query")
        @classmethod
        def check_query(cls, value: str) -> str:
            if not value.strip():
                raise ValueError("빈 질문은 허용되지 않습니다.")
            return value.strip()

    @app.get("/")
    def web_ui():
        return FileResponse(STATIC_DIR / "index.html")

    @app.get("/health")
    def health():
        return {"status":"ok", "model":MODEL_ID, **gpu_status()}

    @app.get("/tools")
    def tools():
        return TOOL_DESCRIPTIONS

    @app.post("/agent")
    def agent(request: AgentRequest):
        try:
            return run_agent(request.query)
        except (RuntimeError, OSError, ValueError) as e:
            raise HTTPException(status_code=503, detail=f"GPU/모델 준비 실패: {e}") from e

    ''', encoding="utf-8")
print("확인:", file_path)


기존 파일 보존 (덮어쓰지 않음): /workspace/05_agent_web_mvp/app/main.py
확인: /workspace/05_agent_web_mvp/app/main.py


## STEP 6. FastAPI 의존성 확인 및 문법 검사

In [ ]:
%pip install -U "fastapi>=0.115,<1" "uvicorn>=0.30,<1" "pydantic>=2,<3" "httpx>=0.27,<1" requests

In [6]:
import ast
for file in (PROJECT_DIR/"app").glob("*.py"):
    ast.parse(file.read_text(encoding="utf-8"))
print("Python 코드 문법 통과")

Python 코드 문법 통과


## STEP 7. 로컬 FastAPI Web UI·API 기본 테스트 — GPU 불필요

In [7]:
from fastapi.testclient import TestClient
# 같은 커널에서 4회차/5회차를 순서대로 열었더라도 올바른 앱을 import합니다.
for module_name in list(sys.modules):
    if module_name == "app" or module_name.startswith("app."):
        sys.modules.pop(module_name, None)
if str(PROJECT_DIR) in sys.path:
    sys.path.remove(str(PROJECT_DIR))
sys.path.insert(0,str(PROJECT_DIR))
from app.main import app

with TestClient(app) as client:
    for path in ("/","/static/app.js","/static/style.css","/health","/tools"):
        response=client.get(path)
        print(path,"→",response.status_code)
        assert response.status_code==200
    bad=client.post("/agent",json={"query":"   "})
    assert bad.status_code==422
print("Web UI 기본 경로 / 입력 검증 확인")

/usr/local/lib/python3.12/dist-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


/ → 200
/static/app.js → 200
/static/style.css → 200
/health → 200
/tools → 200
Web UI 기본 경로 / 입력 검증 확인


## STEP 8. RunPod GPU 서버 실행
###(VS Code 터미널과 RUNPOD JupyterLab 터미널)
```bash
cd /workspace/05_agent_web_mvp
nvidia-smi
python -m pip install -r requirements.txt

# python -m uvicorn app.main:app --host 127.0.0.1 --port 8000 # VS Code SSH 포트 전달(vscode에서 실행할 때 필요)
python -m uvicorn app.main:app --host 0.0.0.0 --port 8000 #RunPod HTTP 프록시로 브라우저 접속
```

## STEP 9. 실제 GPU Agent 기능 테스트
최초 Qwen 모델 로딩에는 시간이 걸립니다. 각 질문은 브라우저 또는 아래 Python 코드로 테스트합니다.

In [8]:
import requests
BASE_URL="http://127.0.0.1:8000"  # Notebook이 RunPod 원격 커널이라면 사용
TEST_CASES=[
  {"test_type":"계산","query":"33명이 30시간씩 GPU를 사용하면 총 몇 GPU-hour야?","expected_tool":"calculator"},
  {"test_type":"글자 수","query":"'AI Agent'의 글자 수를 계산해줘.","expected_tool":"text_length"},
  {"test_type":"현재 시간","query":"지금 한국 날짜와 시간은?","expected_tool":"current_time"},
  {"test_type":"데이터 분석","query":"[10,20,30]의 합계와 평균을 알려줘.","expected_tool":"data_summary"},
  {"test_type":"일반 설명","query":"AI Agent가 무엇인지 한 문장으로 설명해줘.","expected_tool":None},
]
# 서버가 RunPod에서 실행 중일 때만 True로 변경해 실제 모델 평가
RUN_GPU_TESTS=True  # 2026-11-21 직접 변경한 설정: GPU와 Uvicorn이 실행 중일 때만 사용
records=[]
if RUN_GPU_TESTS:
    try:
        health = requests.get(BASE_URL + "/health", timeout=8)
        health.raise_for_status()
        if not health.json().get("cuda_available", False):
            raise RuntimeError("현재 GPU를 사용할 수 없습니다. CPU Pod에서는 Agent 모델 테스트를 진행하지 마세요.")
        print("GPU 상태 확인:", health.json().get("gpu"))
    except Exception as exc:
        raise RuntimeError("FastAPI/GPU 사전 확인 실패. 터미널에서 Uvicorn과 GPU 상태를 점검하세요.") from exc
    for item in TEST_CASES:
        try:
            r=requests.post(BASE_URL+"/agent",json={"query":item["query"]},timeout=240)
            data=r.json()
            success=(r.status_code==200 and data.get("status")=="success" and data.get("tool_name")==item["expected_tool"])
            records.append({**item,"success":success,"status_code":r.status_code,
                    "tool_name":data.get("tool_name"),"elapsed_ms":data.get("elapsed_ms"),"issue":"" if success else str(data)[:180]})
        except requests.RequestException as exc:
            records.append({**item,"success":False,"status_code":None,
                            "tool_name":None,"elapsed_ms":None,"issue":str(exc)[:180]})
else:
    print("실제 GPU 평가 생략: RUN_GPU_TESTS=True로 변경하고, RunPod에서 Uvicorn 서버를 실행하세요.")

GPU 상태 확인: NVIDIA A100 80GB PCIe


## STEP 10. 입력 오류 테스트 — 모델 호출 없음

In [9]:
if RUN_GPU_TESTS:
    bad=requests.post(BASE_URL+"/agent",json={"query":"   "},timeout=30)
    print("빈 질문 HTTP:",bad.status_code)
    records.append({"test_type":"빈 입력","query":"   ","expected_tool":None,
         "success":bad.status_code==422,"status_code":bad.status_code,
         "tool_name":None,"elapsed_ms":None,"issue":"" if bad.status_code==422 else bad.text[:120]})

빈 질문 HTTP: 422


## STEP 11. 실제 기능 평가 CSV 저장
측정하지 않은 결과를 성공으로 기록하지 않습니다. 테스트를 수행한 후 결과를 저장합니다.

In [11]:
import pandas as pd
from IPython.display import display
results_dir=PROJECT_DIR/"results"
results_dir.mkdir(parents=True,exist_ok=True)
columns=["test_type","query","expected_tool","success","status_code","tool_name","elapsed_ms","issue"]
mvp_df=pd.DataFrame(records,columns=columns)
display(mvp_df)
if len(mvp_df):
    out=results_dir/"05_mvp_evaluation.csv"
    mvp_df.to_csv(out,index=False,encoding="utf-8-sig")
    print("실제 테스트 결과 저장:",out)
else:
    print("테스트 미실행: 평가 CSV를 빈 데이터로 만들지 않습니다.")

,test_type,query,expected_tool,success,status_code,tool_name,elapsed_ms,issue
0,계산,33명이 30시간씩 GPU를 사용하면 총 몇 GPU-hour야?,calculator,True,200,calculator,26913.3,
1,글자 수,'AI Agent'의 글자 수를 계산해줘.,text_length,True,200,text_length,654.6,
2,현재 시간,지금 한국 날짜와 시간은?,current_time,False,200,NaN,307.1,"{'status': 'error', 'used_tool': False, 'tool_..."
3,데이터 분석,"[10,20,30]의 합계와 평균을 알려줘.",data_summary,True,200,data_summary,1109.6,
4,일반 설명,AI Agent가 무엇인지 한 문장으로 설명해줘.,NaN,True,200,NaN,583.5,
5,빈 입력,,NaN,True,422,NaN,NaN,


실제 테스트 결과 저장: /workspace/05_agent_web_mvp/results/05_mvp_evaluation.csv


## STEP 12. 결과 개선 / 제출

1. 테스트에서 실패한 항목 하나를 선택하고 원인 계층을 결정합니다(`app.js` / `main.py` / `agent.py` / `tools.py`).
2. 수정 전후 같은 질문을 실행하여 `success`, `tool_name`, `elapsed_ms`를 비교합니다.
3. GitHub에 소스·README·실제 측정 CSV·실행 화면을 제출합니다.

In [ ]:
## STEP 13. 웹서비스 접속 주소 확인

새 터미널을 열고 다음을 입력하면 웹서비스 접속 주소를 확인할 수 있다.
echo "https://${RUNPOD_POD_ID}-8000.proxy.runpod.net"